# 03 · Span과 단계별 실행 추적

목표: 부모·자식 관계, 실행 시간, 동시 요청의 context 격리를 확인한다. 약15분.

**실행:** Python 3.11 공통 .venv 커널에서 위→아래. 다른 Notebook·서버·Docker·모델 설치 없이 독립 실행. 코드는 MOCK 고정 문장·대기만 사용한다. 실제 LLM 품질·처리량 측정이 아니다.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import asyncio
from week10.lab.config import Settings
from week10.lab.experiments import lab_client

## 정상 경로

검색·생성·출력을 assist 부모 작업으로 묶는다.

In [ ]:
async with lab_client() as (app, client):
    r = await client.post("/api/v1/assist", json={"message": "신청 절차"})
    snap = app.state.telemetry.snapshot()
spans = sorted(snap["spans"], key=lambda s: s["offset_ms"])
[(s["name"], s["offset_ms"], s["duration_ms"]) for s in spans]

## 부모·자식 검증

자식 span은 assist의 span_id를 parent_id로 가진다.

In [ ]:
root = next(s for s in spans if s["name"] == "assist")
children = [s for s in spans if s["name"] != "assist"]
assert all(s["parent_id"] == root["span_id"] for s in children)
assert all(s["trace_id"] == root["trace_id"] for s in children)

## 시간 해석

부모 시간은 자식을 포함한다. 부모+자식 합계는 이중 집계다.

In [ ]:
print("부모 실행 ms:", root["duration_ms"])
print("자식 합계 ms:", sum(s["duration_ms"] for s in children))
print("가장 긴 자식:", max(children, key=lambda s: s["duration_ms"])["name"])

## W3C header의 제한된 예시

00 버전·01 flags의 유효한 context만 전달한다. 전체 표준·샘플러·OTLP 구현이 아니다.

In [ ]:
header = "00-" + "a" * 32 + "-" + "b" * 16 + "-01"
async with lab_client() as (app, client):
    r = await client.post(
        "/api/v1/assist", json={"message": "예시"}, headers={"traceparent": header}
    )
    root = next(s for s in app.state.telemetry.spans if s["name"] == "assist")
assert root["trace_id"] == "a" * 32 and root["parent_id"] == "b" * 16

## 잘못된 context

올제로 ID나 지원하지 않는 버전은 새 ID로 시작한다.

In [ ]:
from week10.lab.telemetry import trace_context

for h in ["not-a-header", "00-" + "0" * 32 + "-" + "b" * 16 + "-01"]:
    tid, parent = trace_context(h)
    assert int(tid, 16) > 0 and parent is None
    print(tid)

## 동시 요청 격리

한 context 전역 변수를 공유할 때의 오염을 ContextVar로 피한다.

In [ ]:
async with lab_client(Settings(capacity=2)) as (app, client):
    responses = await asyncio.gather(
        *[client.post("/api/v1/assist", json={"message": str(i)}) for i in range(2)]
    )
    snap = app.state.telemetry.snapshot()
tids = {r.headers["X-Trace-ID"] for r in responses}
assert len(tids) == 2
assert all(len([s for s in snap["spans"] if s["trace_id"] == tid]) == 4 for tid in tids)

## 실패 지점

검색 실패 때 generate span이 생기지 않는 이유를 설명한다.

In [ ]:
async with lab_client() as (app, client):
    await client.post("/api/v1/assist", json={"message": "예시", "scenario": "retrieval_error"})
    spans = list(app.state.telemetry.spans)
assert not any(s["name"] == "generate" for s in spans)
[(s["name"], s["status"]) for s in spans]

## 프로젝트 적용 기록

예상 → 관찰 → 근거 → 다음 변경을 한 줄씩 기록한다. 실제로 실행하지 않은 조건은 미측정으로 표시한다. 학생 정보나 원문 입력은 공개 저장소에 올리지 않는다.